## Imports

In [1]:
import base64  # 바이너리 파일(예: 이미지)을 ASCII 문자열로 인코딩
from src.utils import openai_client

## 웹 상의 이미지에 대한 설명 생성

In [5]:
# https://picsum.photos/200/300 여기에서 보여주는 이미지에 대한 설명
messages = [
    {
        'role': 'user',
        'content': [
            {'type': 'text', 'text': '이 이미지를 설명해줘.'},
            {'type': 'image_url',
             'image_url': {'url': 'https://fastly.picsum.photos/id/915/200/300.jpg?hmac=dzVwmjYlIh3MdKz2l08oFpp1y3XxMyu_1vjGp5Dycd0'}}
        ]
    }
]

In [6]:
response = openai_client.chat.completions.create(
    model='gpt-5.4-mini',
    temperature=0.9,
    messages=messages
)

In [7]:
print(response.choices[0].message.content)

이미지는 위에서 내려다본 듯한 시점의 초록색 식생이 가득한 풍경입니다.  
나무나 관목이 촘촘하게 심어진 듯 보이며, 줄지어 반복되는 패턴이 뚜렷합니다. 전체적으로는 울창한 숲보다는 **정돈된 농장이나 플랜테이션** 같은 느낌을 줍니다.  
초록색이 화면 대부분을 차지해서 매우 자연스럽고 생기 있는 분위기입니다.


## 로컬 이미지 파일의 설명 요청

*   이미지와 같은 바이너리 데이터는 텍스트 형식으로 변환해서 GPT 요청을 보내야 함
*   base64 라이브러리: 이진(binary) 데이터를 ACSII 문자로 인코딩(변환)
    *   전송/저장 호환성 유지
    *   데이터 손상 방지
    *   프로토콜 호환성

In [8]:
def base64encode_image(image_file):
    # 이미지 파일을 이진데이터 읽기 모드로 오픈
    with open(image_file, mode='rb') as f:
        data = f.read()  # 파일 내용 전체를 읽기

        # 이진 데이터를 base64 인코딩을 해서 ASCII 문자열로 변환, utf-8 인코딩
        return base64.b64encode(data).decode(encoding='utf-8')

In [9]:
# 이미지 파일(jpg)들이 저장된 경로
image_1 = './images/image_1.jpg'
image_2 = './images/image_2.jpg'

In [10]:
b64_image_1 = base64encode_image(image_1)

In [12]:
b64_image_1[:100]
#> 이미지 이진 데이터가 ASCII 문자열로 인코딩

'/9j/4QDeRXhpZgAASUkqAAgAAAAGABIBAwABAAAAAQAAABoBBQABAAAAVgAAABsBBQABAAAAXgAAACgBAwABAAAAAgAAABMCAwAB'

In [13]:
b64_image_1[-100:]

'AAAAAAAAAAAQAAAAABQAAAAAEQAICAAAAQAAQAAAACCAAABAAAAEAAAAECAAQARsAAAIQAADAAAAAAAAAAAABAAAAEgAAAAA/9k='

In [14]:
b64_image_2 = base64encode_image(image_2)

In [18]:
# 로컬 이미지 데이터를 GPT 서버로 전송
messages = [{
    'role': 'user',
    'content': [
        {'type': 'text', 'text': '이 이미지에 대해 설명해줘.'},
        {'type': 'image_url', 'image_url': {'url': f'data:image/jpg;base64,{b64_image_2}'}}
    ]
}]

In [19]:
response = openai_client.chat.completions.create(
    model='gpt-5.4-mini',
    messages=messages,
    temperature=0.9
)
print(response.choices[0].message.content)

이 이미지는 **분홍빛~코랄색의 꽃들이 클로즈업된 사진**입니다.  
꽃잎은 두껍고 부드러워 보이며, 가운데에는 **노란 수술**이 보여서 꽃의 디테일이 잘 드러납니다.

전체적으로는:

- **따뜻한 색감**이 강하고
- **부드러운 조명**이 들어와 몽환적인 분위기를 만들며
- 배경은 흐릿하게 처리되어 있어 **꽃에 시선이 집중**됩니다.

아마도 **실내에서 촬영한 꽃 사진**처럼 보이고, 장식용 꽃병에 꽂힌 꽃들로 보입니다.  
원하시면 제가 이 이미지를 **더 짧게 한 문장으로 요약**해드리거나, **감성적인 느낌으로 설명**해드릴 수도 있어요.


In [22]:
# 2개의 이미지를 비교를 요청하는 메시지 프롬프트 작성

messages = [{
    'role': 'user',
    'content': [
        {'type': 'text', 'text': '두 개의 이미지를 비교해줘.'},
        {'type': 'image_url', 'image_url': {'url': f'data:image/jpg;base64,{b64_image_1}'}},
        {'type': 'image_url', 'image_url': {'url': f'data:image/jpg;base64,{b64_image_2}'}}
    ]
}]


In [23]:
response = openai_client.chat.completions.create(
    model='gpt-5.4-mini',
    messages=messages,
    temperature=0.9
)
print(response.choices[0].message.content)

두 이미지는 분위기와 주제가 크게 다릅니다.

### 1) 첫 번째 이미지
- **주제**: 밤의 고속도로/도로 풍경
- **특징**: 자동차 불빛이 긴 선처럼 남아 있는 **장노출 사진**
- **색감**: 짙은 파란 밤하늘과 흰색·빨간색 빛의 대비가 강함
- **느낌**: 역동적이고 도시적이며, 빠른 움직임과 에너지가 느껴짐

### 2) 두 번째 이미지
- **주제**: 꽃 클로즈업
- **특징**: 여러 송이의 꽃을 가까이서 찍은 **부드러운 매크로/접사 느낌**
- **색감**: 분홍, 주황, 노랑 계열의 따뜻한 색이 중심
- **느낌**: 아늑하고 부드럽고 감성적이며, 첫 번째보다 훨씬 정적인 분위기

### 공통점
- 둘 다 **빛을 인상적으로 활용**한 사진입니다.
- 첫 번째는 **빛의 궤적**, 두 번째는 **꽃에 비친 따뜻한 조명**이 핵심입니다.
- 시각적으로 강한 인상을 주는 **예술적인 사진**이라는 점도 비슷합니다.

### 한줄 요약
- **첫 번째는 “움직임과 속도”**,  
- **두 번째는 “따뜻함과 섬세함”**을 보여주는 이미지입니다.

원하시면 제가 이 두 이미지를 **구도, 색감, 감정, 촬영기법** 기준으로 더 자세히 표로 비교해드릴게요.
